# Step 4: Count puncta in individual cells (3D)

Needs `Step3_ImageFolderAnalysis.ipynb` output (`spot_analysis.csv` + `*_cellMask.tiff`
files in the `<folder>_analysis` directory).

This assigns oligomer puncta to individual cells using the full 3D cell segmentation
(`RASP.count_puncta_in_individual_cells_threshold_3D`), giving per-cell identities
and a puncta-per-cell count. Cell mask cleanup (removing objects that are too small,
too large, don't span enough z-planes, or fill an implausible fraction of a plane) is
handled automatically inside this call via `threshold_cell_areas_3d` -- you do not need
to pre-clean the `*_cellMask.tiff` files yourself. If you want to visually tune the
`lower_cell_size_threshold` / `upper_cell_size_threshold` values before committing to a
full run, see `Optional_CellMask_SizeThresholding.ipynb` in this folder.

In [ ]:
import sys

sys.path.append("..")  # Adds higher directory to python modules path.
import os
import numpy as np
import polars as pl
from src import RASPRoutines

RASP = RASPRoutines.RASP_Routines()

## Configuration

`restrict_to_bulk_stain` is the on/off switch this notebook is built around:

- `True` (default): only puncta whose `overlap_with_bulk_stain` flag is `True`
  (i.e. that fell inside the bulk-stain mask in Step3) are assigned to cells.
- `False`: every detected punctum is assigned to cells, regardless of bulk-stain
  overlap.

This requires `spot_analysis.csv` to have been produced with `bulk_string` set in
`Step3_ImageFolderAnalysis.ipynb` (so the `overlap_with_bulk_stain` column exists).

In [ ]:
restrict_to_bulk_stain = True  # default True -- see markdown above

# This repo doesn't ship a real three-channel example, so this points at the
# example_images_analysis folder produced by Step3_ImageFolderAnalysis.ipynb
# (which used bulk_string="C0" as a stand-in). Replace with your own folder.
folder = os.path.abspath(r"../example_images")
analysis_folder = folder + "_analysis"
analysis_file = os.path.join(analysis_folder, "spot_analysis.csv")

protein_string = "C1"
cell_string = "C0"

threshold_lower = 0.0  # photon threshold; puncta with sum_intensity_in_photons
# above this are counted
lower_cell_size_threshold = (
    2000.0  # voxels; tune against Optional_CellMask_SizeThresholding.ipynb
)
upper_cell_size_threshold = np.inf

In [ ]:
analysis_data = pl.read_csv(analysis_file)

if restrict_to_bulk_stain:
    if "overlap_with_bulk_stain" not in analysis_data.columns:
        raise ValueError(
            "restrict_to_bulk_stain=True but 'overlap_with_bulk_stain' is missing from "
            f"{analysis_file}. Re-run Step3_ImageFolderAnalysis.ipynb with bulk_string set "
            "to produce it, or set restrict_to_bulk_stain=False to analyse all puncta."
        )
    analysis_data = analysis_data.filter(pl.col("overlap_with_bulk_stain"))

print(
    f"{len(analysis_data)} puncta going into cell assignment "
    f"(restrict_to_bulk_stain={restrict_to_bulk_stain})"
)

## Run the 3D per-cell puncta count

Produces one row per detected cell (per image), with the number of puncta assigned to
it (`n_puncta_in_cell`), cell volume, and centroid -- written to a CSV under
`<analysis_folder>/New_Cell_Analysis/`.

In [ ]:
end_string = "bulkrestricted" if restrict_to_bulk_stain else "allpuncta"

cell_punctum_analysis = RASP.count_puncta_in_individual_cells_threshold_3D(
    analysis_data,
    analysis_file,
    threshold_lower=threshold_lower,
    cell_string=cell_string,
    protein_string=protein_string,
    lower_cell_size_threshold=lower_cell_size_threshold,
    upper_cell_size_threshold=upper_cell_size_threshold,
    dims=3,
    sigma1=2.0,
    sigma2=40.0,
    spacing=(0.5, 0.11, 0.11),
    replace_files=True,
    end_string=end_string,
)
cell_punctum_analysis

In [ ]:
# quick summary: mean puncta per cell, and how many cells were found in total
if isinstance(cell_punctum_analysis, pl.DataFrame) and len(cell_punctum_analysis) > 0:
    print("n cells:", len(cell_punctum_analysis))
    print("mean puncta per cell:", cell_punctum_analysis["n_puncta_in_cell"].mean())
else:
    print("No cells found -- check paths/thresholds above.")